In [2]:
import pandas as pd

df = pd.read_csv("source.csv")


In [3]:

df["delta"] = (df["close"] - df["open"]) / df["open"]

In [4]:
import numpy as np

statLength = 100

def _rolling_zscore_last(x):
    std = np.std(x)
    if std == 0 or np.isnan(std):
        return np.nan
    return (x[-1] - np.mean(x)) / std

df.loc[:, "price_effect"] = (df["high"] - df["low"]) * 0.25 + abs(df["close"] - df["open"]) * 0.75
df.loc[:, "vol_min"] = df["volume"].rolling(window=statLength, min_periods=statLength).min()
# avoid division by zero
df.loc[:, "vol_min"] = df.loc[:, "vol_min"].replace(0, np.nan)
df.loc[:, "vol_norm"] = df["volume"] / df.loc[:, "vol_min"]
df.loc[:, "pricePerVol"] = df.loc[:, "price_effect"] / df.loc[:, "vol_norm"]
# compute z-score within a rolling window and keep the z-score of the last element in each window
df.loc[:, "sentiment_zs"] = df["pricePerVol"].rolling(window=statLength, min_periods=statLength).apply(lambda x: _rolling_zscore_last(x), raw=True)


In [7]:

col = ["volume", "delta", "sentiment_zs", "takerbuyvolume"]
df.to_csv("target.csv", columns=col, index=False)
df.tail()

,opentime,open,high,low,close,volume,closetime,quoteassetvolume,numberoftrades,takerbuyvolume,takerbuyquotevolume,ignore,delta,price_effect,vol_min,vol_norm,pricePerVol,sentiment_zs
4995,1772575200000,67976.5,68485.5,67976.5,68275.1,3828.395,1772578799999,2.614342e+08,137019,1757.102,1.199804e+08,0,0.004393,351.20,1986.768,1.926946,182.257296,2.149504
4996,1772578800000,68275.2,68821.5,68206.4,68295.3,5639.779,1772582399999,3.863490e+08,160206,3120.412,2.137864e+08,0,0.000294,168.85,1986.768,2.838670,59.482078,-0.871602
4997,1772582400000,68295.2,68383.4,67826.0,68065.8,6011.630,1772585999999,4.094289e+08,209192,2889.420,1.968689e+08,0,-0.003359,311.40,1986.768,3.025834,102.913778,0.199538
4998,1772586000000,68065.8,68887.9,67922.4,68377.9,11304.590,1772589599999,7.732029e+08,300236,6108.492,4.179196e+08,0,0.004585,475.45,1986.768,5.689940,83.559762,-0.276254
4999,1772589600000,68384.6,68419.7,67950.0,68288.1,5287.822,1772593199999,3.604486e+08,201445,2504.570,1.707246e+08,0,-0.001411,189.80,1986.768,2.661520,71.312644,-0.574735
